# 061 — The LSTM's Core Idea

Lesson 060 diagnosed the disease: every step **multiplies** the gradient by
something below 1. The LSTM's answer is a path through time whose default
operation is **addition** — the cell state — with three gates deciding when to
change it.

| Part | What we check |
|---|---|
| A | multiply-every-step against add-every-step |
| B | the carry: $C_t = f_t C_{t-1} + i_t \tilde{C}_t$ |
| C | the decay is a product of forget gates, so it is learnable |
| D | what the three gates cost: exactly 4x a SimpleRNN |

## Part A — The shape of the problem

In [ ]:
import numpy as np

steps = 100
for factor in (0.5, 0.9, 0.99):
    print(f"  multiplying by {factor} each step -> {factor ** steps:.3e} after {steps}")
for keep in (1.0,):
    print(f"  multiplying by {keep} each step -> {keep ** steps:.3f}  <- the only "
          f"safe base")
assert 0.9 ** 100 < 1e-4
print()
print("Any base below 1 dies exponentially; 1.0 survives exactly. So the")
print("question is not 'how do we stop the decay' but 'can the network CHOOSE")
print("a base near 1?' A tanh derivative can never be chosen. A gate can.")

## Part B — The carry, by hand

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

H = 4
rng = np.random.default_rng(0)
C = np.zeros(H)
candidate = rng.normal(0, 1, H)

print("  a slot with the forget gate wide open and the input gate shut:")
C_open = np.array([1.0, -1.0, 0.5, 0.0])
for t in range(5):
    C_open = 1.0 * C_open + 0.0 * candidate      # f=1, i=0: nothing happens
print(f"    after 5 steps: {C_open.round(3)}   (unchanged)")
assert np.allclose(C_open, [1.0, -1.0, 0.5, 0.0])

print("  the same slot with the forget gate half closed:")
C_half = np.array([1.0, -1.0, 0.5, 0.0])
for t in range(5):
    C_half = 0.5 * C_half + 0.0 * candidate
print(f"    after 5 steps: {C_half.round(4)}   (down by {0.5 ** 5:.4f}x)")
assert np.allclose(C_half, np.array([1.0, -1.0, 0.5, 0.0]) * 0.5 ** 5)
print()
print("Addition with a gate of 1 is an identity path through time. That is the")
print("whole invention - and the hidden state h_t = o_t * tanh(C_t) is a")
print("SEPARATE, filtered view of it, which is the second idea: what to")
print("remember and what to use right now are different questions.")

## Part C — The gradient along the cell state

In [ ]:
H, F = 20, 8

def cell_state_flow(steps, forget_bias, seed=0):
    """Gradient reaching the cell state, and the mean forget-gate value."""
    r = np.random.default_rng(seed)
    W = {g: r.normal(0, 0.3, (F + H, H)) for g in "fico"}
    b = {g: np.zeros(H) for g in "fico"}
    b["f"] = b["f"] + forget_bias

    X = r.normal(0, 1, (steps, F))
    h, C, gates = np.zeros(H), np.zeros(H), []
    for t in range(steps):
        z = np.concatenate([h, X[t]])
        f = sigmoid(z @ W["f"] + b["f"])
        i = sigmoid(z @ W["i"] + b["i"])
        cand = np.tanh(z @ W["c"] + b["c"])
        o = sigmoid(z @ W["o"] + b["o"])
        C = f * C + i * cand                 # addition, not multiplication
        h = o * np.tanh(C)
        gates.append(f)

    dC, norms = np.ones(H), []
    for t in range(steps - 1, -1, -1):
        norms.append(np.linalg.norm(dC))
        dC = dC * gates[t]                   # the ENTIRE backward path
    return np.array(norms[::-1]), float(np.mean([g.mean() for g in gates]))

print(f"  {'forget bias':>12}{'mean f':>9}{'99 steps back':>16}")
for bias in (0.0, 1.0, 2.0, 3.0):
    norms, mean_f = cell_state_flow(100, bias)
    print(f"  {bias:>12.1f}{mean_f:>9.3f}{norms[0]:>16.3e}")

print("\n  and why: the decay is f_bar raised to the number of steps")
for mean_f in (0.5, 0.9, 0.99, 0.999):
    print(f"    f_bar = {mean_f:<6} after 100 steps: {mean_f ** 100:.3e}")
assert 0.999 ** 100 > 0.9
print()
print("Still an exponential. But the base is a number the network computes,")
print("and it can drive it towards 1. It could never enlarge tanh'.")

## Part D — What the gates cost

In [ ]:
for f_dim, h_dim in ((5, 3), (2, 32), (100, 128)):
    rnn = (f_dim + h_dim) * h_dim + h_dim
    print(f"  F={f_dim:<4} H={h_dim:<4} SimpleRNN {rnn:>9,}   LSTM {4 * rnn:>9,}")
    assert 4 * rnn == 4 * ((f_dim + h_dim) * h_dim + h_dim)
print()
print("Exactly 4x, always: three gates plus the candidate, each with the same")
print("shape as a SimpleRNN's single weight block. Four times the parameters")
print("for a decay rate the network controls - that is the trade.")

## What this notebook established

- Any per-step factor below 1 dies exponentially; only a base at 1 survives, so
  the real question is whether the network can **choose** the base.
- The cell state updates by addition, $C_t = f_t C_{t-1} + i_t \tilde{C}_t$, so
  with $f = 1$ and $i = 0$ a slot is carried unchanged — measured over 5 steps.
- The backward path along the cell state is a **product of forget gates**, so
  the decay rate is learnable: $\bar{f} = 0.999$ leaves over 90% after 100
  steps.
- The hidden state is a filtered view of the cell state, which separates what to
  remember from what to use now.
- An LSTM costs **exactly 4x** a SimpleRNN, at every size.

## Exercises

1. Set the input gate to 1 and the forget gate to 1 and run 20 steps. What
   happens to the magnitude of the cell state, and why is $\tanh$ in $h_t$
   useful?
2. Measure the mean forget gate at bias 5. Is more always better?
3. Replace the addition with multiplication and re-measure Part C. How many
   steps of memory are left?